# Message
Configuration: `stages/01_message_dynamics/config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
message_experiment_dir = ROOT / 'experiment/research/bp_failure_dynamics/stages/01_message_dynamics'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

message_data_paths = ExperimentPaths(ROOT, message_experiment_dir / 'config/paths.json')


# BP message dynamics experiments

This notebook tests message-level diagnostics of BP success and failure. The central hypothesis is treated as falsifiable: logical failure may correspond to persistent message instability, or to a stable syndrome-valid fixed point with the wrong logical label.

The **benchmark result** is the first syndrome-valid BP decision. The **diagnostic trajectory** always runs for 50 iterations with early stopping disabled. Tanner variables represent DEM fault mechanisms, not necessarily individual physical-qubit errors. CSV output is written under `experiment/research/bp_failure_dynamics/stages/01_message_dynamics/`; plots are only displayed inline.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5
from noises_ca import CircuitBuilder, ErrorRate

message_config = json.loads((message_experiment_dir / 'config/message.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# diagnostic_iterations: at least 3.
# benchmark_iterations: positive.
# ms_scaling_factor: positive.
# late_fraction: in (0, 1].
# message_zero_tolerance: non-negative.
# top_edge_count: positive.
# hidden_belief_threshold: non-negative.
# hidden_message_threshold: positive.
# unstable_normalized_residual_floor: positive.
# unstable_edge_quantile: in (0, 1).
# Expected relationship: candidate_scan_limit >= 2 * cases_per_class.
message_data = message_data_paths.run()

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(message_data_paths, message_config)

message_input = message_data_paths.input('error_weight')
message_stats = message_data_paths.stats()
message_trace = message_data_paths.trace()
message_stats.mkdir(parents=True, exist_ok=True)
message_trace.mkdir(parents=True, exist_ok=True)

def message_log(level, *items):
    if message_config['print_level'] >= level:
        print(*items)

sns.set_theme(style='whitegrid')
message_log(2, {'root': str(ROOT), 'csv_output': str(message_experiment_dir)})


In [ ]:
message_dtype = {
    'p': float,
    'err': str,
    'det': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

message_weight2 = pd.read_csv(message_data_paths.input('weight_2_data_csv'), dtype=message_dtype)
message_weight3 = pd.read_csv(message_data_paths.input('weight_3_data_csv'), dtype=message_dtype)

for message_frame in (message_weight2, message_weight3):
    message_frame['syndrome_weight'] = message_frame.det.str.count('1').astype(int)
    message_frame['stored_bposd_failure'] = message_frame.pred_b != message_frame.true

pd.DataFrame([{'weight2_rows': len(message_weight2), 'weight3_rows': len(message_weight3)}])


## Instrumented parallel minimum-sum

Index 0 is initialization (`c2v=0`, `v2c=prior`, `LLR=prior`). Index $t>0$ is the state after update $t$. Sign flips use a zero dead zone, and cross-$p$ comparisons include residual normalized by the variable's prior-LLR magnitude.

The following helpers expose directed edge messages and posterior beliefs from native BP. They define the residual and instability measurements used by the next comparisons; run these definitions before tracing any case.


In [ ]:
from experiment.utils import bp_message_dynamics

message_Context = bp_message_dynamics.Context
message_MessageTrace = bp_message_dynamics.MessageTrace
message_cache = {}

def message_context_for(p):
    return bp_message_dynamics.context_for(
        p,
        CACHE=message_cache,
        CA_9_2_5=CA_9_2_5,
        CircuitBuilder=CircuitBuilder,
        Context=message_Context,
        ErrorRate=ErrorRate,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
    )

from experiment.utils import misc

message_bit_array = misc.bit_array
message_bit_text = misc.bit_text
message_logical = bp_message_dynamics.logical

def message_trace_min_sum(
    context,
    syndrome,
    iterations,
    stop_early=False,
):
    return bp_message_dynamics.trace_min_sum(context, syndrome, iterations, stop_early, CONFIG=message_config, MessageTrace=message_MessageTrace)

def message_edge_metrics(context, trace, direction):
    return bp_message_dynamics.message_edge_metrics(context, trace, direction, CONFIG=message_config)

def message_variable_metrics(context, trace, edges):
    return bp_message_dynamics.message_variable_metrics(context, trace, edges, CONFIG=message_config)


In [ ]:
from ldpc import BpDecoder

message_validation = message_weight2.drop_duplicates(['p', 'det']).sample(12, random_state=message_config['random_seed'])
for message_row in message_validation.itertuples(index=False):
    ctx = message_context_for(message_row.p)
    message_syndrome = message_bit_array(message_row.det)
    tr = message_trace_min_sum(ctx, message_syndrome, message_config['benchmark_iterations'], stop_early=True)
    message_native = BpDecoder(
        ctx.H,
        error_channel=ctx.priors,
        max_iter=message_config['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=message_config['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    np.testing.assert_array_equal(tr.hard[-1], np.asarray(message_native.decode(message_syndrome), np.uint8))
    np.testing.assert_allclose(tr.llr[-1], ctx.prior_llr + np.bincount(ctx.edge_variables, weights=tr.c2v[-1], minlength=ctx.H.shape[1]))

message_log(2, {'native_validation_cases': len(message_validation)})
weight2_predictions = {}
for message_row in tqdm(message_weight2.drop_duplicates(['p', 'det']).itertuples(index=False), desc='Audit weight-2 pure BP', disable=message_config['print_level'] == 1):
    ctx = message_context_for(message_row.p)
    tr = message_trace_min_sum(ctx, message_bit_array(message_row.det), message_config['benchmark_iterations'], stop_early=True)
    weight2_predictions[message_row.p, message_row.det] = {
        'prediction': message_bit_text(message_logical(ctx, tr.hard[-1])),
        'syndrome_consistent': bool(tr.syndrome_consistent[-1]),
        'iterations': len(tr.llr) - 1,
    }

weight2_eval = message_weight2.copy()
weight2_eval['pure_bp_prediction'] = [weight2_predictions[row.p, row.det]['prediction'] for row in weight2_eval.itertuples()]
weight2_eval['pure_bp_syndrome_consistent'] = [weight2_predictions[row.p, row.det]['syndrome_consistent'] for row in weight2_eval.itertuples()]
weight2_eval['pure_bp_iterations'] = [weight2_predictions[row.p, row.det]['iterations'] for row in weight2_eval.itertuples()]
weight2_eval['pure_bp_logical_failure'] = weight2_eval.pure_bp_prediction != weight2_eval.true
weight2_fail = weight2_eval[weight2_eval.pure_bp_syndrome_consistent & weight2_eval.pure_bp_logical_failure]
audit = pd.DataFrame([{
    'rows': len(weight2_eval),
    'syndrome_consistent_bp_failures': len(weight2_fail),
    'unique_failure_supports': weight2_fail.err.nunique(),
    'unique_failure_syndromes': weight2_fail.det.nunique(),
    'unique_failure_inputs': len(weight2_fail[['p', 'det', 'true']].drop_duplicates()),
}])

audit.to_csv(message_data_paths.output('weight2_data_audit_csv'), index=False)
failure_row = weight2_fail.iloc[0]
controls = weight2_eval[~weight2_eval.pure_bp_logical_failure & weight2_eval.pure_bp_syndrome_consistent & np.isclose(weight2_eval.p, failure_row.p) & (weight2_eval.syndrome_weight == failure_row.syndrome_weight)].drop_duplicates(['p', 'det', 'true'])
assert len(controls) > 0
success_row = controls.iloc[0]
representatives = {}

for name, message_row in {'failure': failure_row, 'success': success_row}.items():
    ctx = message_context_for(message_row.p)
    tr = message_trace_min_sum(ctx, message_bit_array(message_row.det), message_config['diagnostic_iterations'])
    c = message_edge_metrics(ctx, tr, 'c2v')
    v = message_edge_metrics(ctx, tr, 'v2c')
    vm = message_variable_metrics(ctx, tr, c)
    representatives[name] = {
        'row': message_row,
        'context': ctx,
        'trace': tr,
        'c2v': c,
        'v2c': v,
        'variables': vm,
    }
    history = []

    for direction in ('c2v', 'v2c'):
        message_values = getattr(tr, direction)
        for t in range(message_values.shape[0]):
            history.extend(({
                'case': name,
                'iteration': t,
                'direction': direction,
                'edge_id': e,
                'check_id': ctx.edge_checks[e],
                'variable_id': ctx.edge_variables[e],
                'message': message_values[t, e],
            } for e in range(message_values.shape[1])))

    pd.DataFrame(history).to_csv(message_data_paths.output('name__message_history_csv', name=name), index=False)
    pd.concat([c, v]).to_csv(message_data_paths.output('name__edge_metrics_csv', name=name), index=False)
    vm.to_csv(message_data_paths.output('name__variable_metrics_csv', name=name), index=False)

rep_summary = pd.DataFrame([{
    'case': name,
    'p': x['row'].p,
    'syndrome': x['row'].det,
    'true': x['row'].true,
    'first_valid_iteration': x['trace'].first_valid_iteration,
    'first_valid_prediction': message_bit_text(message_logical(x['context'], x['trace'].hard[x['trace'].first_valid_iteration])),
    'diagnostic_final_prediction': message_bit_text(message_logical(x['context'], x['trace'].hard[-1])),
    'diagnostic_final_syndrome_consistent': x['trace'].syndrome_consistent[-1],
} for name, x in representatives.items()])

rep_summary.to_csv(message_data_paths.output('representative_case_summary_csv'), index=False)
rep_summary


In [ ]:
message_fig, message_axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
for r, (name, message_x) in enumerate(representatives.items()):
    for c, direction in enumerate(('c2v', 'v2c')):
        top = message_x[direction].nlargest(message_config['top_edge_count'], 'late_residual').edge_id.to_numpy()
        message_values = getattr(message_x['trace'], direction)[1:, top].T
        bound = max(np.abs(message_values).max(), np.finfo(float).eps)

        sns.heatmap(
            message_values,
            ax=message_axes[r, c],
            cmap='coolwarm',
            center=0,
            vmin=-bound,
            vmax=bound,
            yticklabels=[f'e{i}' for i in top],
        )
        message_axes[r, c].set(title=f'{name}: {direction}', xlabel='iteration', ylabel='top late-residual edge')

plt.show()
message_fig, message_axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for message_ax, (name, message_x) in zip(message_axes, representatives.items()):
    for e in message_x['c2v'].nlargest(8, 'late_residual').edge_id:
        message_ax.plot(message_x['trace'].c2v[1:, e], label=f"c{message_x['context'].edge_checks[e]}→v{message_x['context'].edge_variables[e]}")
    message_ax.set(title=f'{name}: unstable c→v trajectories', xlabel='iteration', ylabel='LLR')
    message_ax.legend(fontsize=8, ncol=2)

plt.show()


## Experiment 2 — matched pure-BP outcomes

Weight 2 has only six unique failing decoder inputs, so the comparison uses weight 3. Cases are deduplicated by `(p, syndrome, true label)`, pure BP is recomputed without using an OSD fallback label, and failure/success groups are matched exactly on `(p, syndrome weight, syndrome validity)`. Syndrome-valid logical-wrong and syndrome-invalid outcomes are reported separately.

The next cells reconstruct the matched cohorts and compare message-level summaries for pure-BP success and failure. Read the comparison within its selected weight and deduplicated decoder-input population, rather than as a physical-noise failure rate.


In [ ]:
pool = message_weight3.drop_duplicates(['p', 'det', 'true']).sample(frac=1, random_state=message_config['random_seed']).head(message_config['candidate_scan_limit'])
classified = []
for message_row in tqdm(pool.itertuples(index=False), total=len(pool), desc='Classify pure BP', disable=message_config['print_level'] == 1):
    ctx = message_context_for(message_row.p)
    tr = message_trace_min_sum(ctx, message_bit_array(message_row.det), message_config['benchmark_iterations'], stop_early=True)
    pred = message_logical(ctx, tr.hard[-1])
    classified.append({
        'p': message_row.p,
        'det': message_row.det,
        'true': message_row.true,
        'err': message_row.err,
        'syndrome_weight': message_row.syndrome_weight,
        'bp_logical_failure': int(not np.array_equal(pred, message_bit_array(message_row.true))),
        'bp_syndrome_consistent': int(tr.syndrome_consistent[-1]),
        'bp_iterations': len(tr.llr) - 1,
        'benchmark_prediction': message_bit_text(pred),
    })

classified = pd.DataFrame(classified)
message_paired_rows = []
message_pair_id = 0

for message_, message_group in classified.groupby(['p', 'syndrome_weight', 'bp_syndrome_consistent']):
    f = message_group[message_group.bp_logical_failure == 1].sample(frac=1, random_state=message_config['random_seed']).reset_index(drop=True)
    s = message_group[message_group.bp_logical_failure == 0].sample(frac=1, random_state=message_config['random_seed'] + 1).reset_index(drop=True)
    n = min(len(f), len(s))

    for i in range(n):
        message_paired_rows += [{**f.iloc[i].to_dict(), 'matched_pair_id': message_pair_id}, {**s.iloc[i].to_dict(), 'matched_pair_id': message_pair_id}]
        message_pair_id += 1

matched_all = pd.DataFrame(message_paired_rows)
available = matched_all.matched_pair_id.nunique()
take = min(message_config['cases_per_class'], available)
assert take > 0
selected_pair_ids = pd.Series(matched_all.matched_pair_id.unique()).sample(take, random_state=message_config['random_seed']).to_numpy()
message_matched = matched_all[matched_all.matched_pair_id.isin(selected_pair_ids)].sample(frac=1, random_state=message_config['random_seed'])
assert message_matched.groupby(['p', 'syndrome_weight', 'bp_syndrome_consistent', 'bp_logical_failure']).size().unstack(fill_value=0).nunique(axis=1).eq(1).all()
message_matched['trajectory_id'] = message_matched.apply(lambda row: f'p={row.p:.17g}|s={row.det}', axis=1)
message_log(2, {'matched_cases_per_class': take})
message_records = []

for message_sample_id, message_row in enumerate(tqdm(message_matched.itertuples(index=False), total=len(message_matched), desc='Message metrics', disable=message_config['print_level'] == 1)):
    ctx = message_context_for(message_row.p)
    tr = message_trace_min_sum(ctx, message_bit_array(message_row.det), message_config['diagnostic_iterations'])
    em = message_edge_metrics(ctx, tr, 'c2v')
    vm = message_variable_metrics(ctx, tr, em)
    hidden = (vm.normalized_belief_instability <= message_config['hidden_belief_threshold']) & (vm.normalized_message_instability >= message_config['hidden_message_threshold'])
    diagnostic_first_prediction = None if tr.first_valid_iteration is None else message_bit_text(message_logical(ctx, tr.hard[tr.first_valid_iteration]))
    message_records.append({
        'sample_id': message_sample_id,
        'p': message_row.p,
        'error_weight': 3,
        'error_support': message_row.err,
        'syndrome_weight': message_row.syndrome_weight,
        'syndrome': message_row.det,
        'true': message_row.true,
        'matched_pair_id': message_row.matched_pair_id,
        'trajectory_id': message_row.trajectory_id,
        'bp_logical_failure': message_row.bp_logical_failure,
        'bp_syndrome_consistent': message_row.bp_syndrome_consistent,
        'benchmark_iterations': message_row.bp_iterations,
        'benchmark_prediction': message_row.benchmark_prediction,
        'diagnostic_first_valid_iteration': tr.first_valid_iteration,
        'diagnostic_first_valid_prediction': diagnostic_first_prediction,
        'diagnostic_final_syndrome_consistent': int(tr.syndrome_consistent[-1]),
        'diagnostic_final_prediction': message_bit_text(message_logical(ctx, tr.hard[-1])),
        'max_late_residual': em.late_residual.max(),
        'mean_late_residual': em.late_residual.mean(),
        'max_normalized_late_residual': em.normalized_late_residual.max(),
        'mean_normalized_late_residual': em.normalized_late_residual.mean(),
        'max_flip_rate': em.sign_flip_rate.max(),
        'mean_flip_rate': em.sign_flip_rate.mean(),
        'max_period2_strength': em.period2_strength.max(),
        'max_cancellation': vm.late_cancellation.max(),
        'mean_late_cancellation': vm.late_cancellation.mean(),
        'hidden_variable_count': int(hidden.sum()),
        'hidden_variable_fraction': hidden.mean(),
        'mean_belief_instability': vm.belief_instability.mean(),
        'mean_message_instability': vm.message_instability.mean(),
        'mean_extrinsic_conflict': vm.late_extrinsic_conflict.mean(),
    })

sample_metrics = pd.DataFrame(message_records)
sample_metrics.to_csv(message_data_paths.output('matched_sample_metrics_csv'), index=False)
trajectory_multiplicity = sample_metrics.groupby('trajectory_id').size().rename('trajectory_multiplicity')
sample_metrics = sample_metrics.join(trajectory_multiplicity, on='trajectory_id')

sample_metrics.to_csv(message_data_paths.output('matched_sample_metrics_csv'), index=False)
metric_cols = ['max_late_residual', 'mean_late_residual', 'max_normalized_late_residual', 'mean_normalized_late_residual', 'max_flip_rate', 'mean_flip_rate', 'max_period2_strength', 'max_cancellation', 'mean_late_cancellation', 'hidden_variable_count', 'hidden_variable_fraction', 'mean_belief_instability', 'mean_message_instability', 'mean_extrinsic_conflict']
summary = sample_metrics.groupby(['bp_syndrome_consistent', 'bp_logical_failure'])[metric_cols].agg(['count', 'mean', 'median', 'std']).reset_index()
summary.columns = ['_'.join((str(x) for x in col if str(x))) if isinstance(col, tuple) else col for col in summary.columns]

summary.to_csv(message_data_paths.output('failure_success_group_summary_csv'), index=False)
message_trajectory_summary = sample_metrics.groupby(['trajectory_id', 'p', 'syndrome'], as_index=False).agg(
    observations=('sample_id', 'size'),
    failure_fraction=('bp_logical_failure', 'mean'),
    syndrome_consistent=('bp_syndrome_consistent', 'first'),
    max_normalized_late_residual=('max_normalized_late_residual', 'first'),
    max_flip_rate=('max_flip_rate', 'first'),
)
message_trajectory_summary.to_csv(message_data_paths.output('unique_trajectory_metrics_csv'), index=False)
summary


In [ ]:
plot_data = sample_metrics.assign(outcome=np.select([(sample_metrics.bp_syndrome_consistent == 1) & (sample_metrics.bp_logical_failure == 0), (sample_metrics.bp_syndrome_consistent == 1) & (sample_metrics.bp_logical_failure == 1), (sample_metrics.bp_syndrome_consistent == 0) & (sample_metrics.bp_logical_failure == 0)], ['valid-correct', 'valid-logical-wrong', 'invalid-logical-match'], default='invalid-logical-mismatch'))
message_fig, message_axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
for message_ax, message_metric in zip(message_axes, ['max_normalized_late_residual', 'max_flip_rate', 'max_period2_strength']):
    sns.boxplot(
        data=plot_data,
        x='outcome',
        y=message_metric,
        ax=message_ax,
        showfliers=False,
    )
    sns.stripplot(
        data=plot_data,
        x='outcome',
        y=message_metric,
        ax=message_ax,
        color='black',
        alpha=0.12,
        size=2,
    )

plt.show()
rows = []
for syndrome_consistent, cohort in sample_metrics.groupby('bp_syndrome_consistent'):
    message_thresholds = np.unique(np.quantile(cohort.max_normalized_late_residual, np.linspace(0, 0.95, 30)))
    for message_threshold in message_thresholds:
        message_selected = cohort[cohort.max_normalized_late_residual > message_threshold]
        if len(message_selected):
            rows.append({
                'bp_syndrome_consistent': syndrome_consistent,
                'threshold': message_threshold,
                'cases': len(message_selected),
                'matched_failure_fraction': message_selected.bp_logical_failure.mean(),
            })

enrichment = pd.DataFrame(rows)
enrichment.to_csv(message_data_paths.output('matched_failure_enrichment_csv'), index=False)
valid_enrichment = enrichment[enrichment.bp_syndrome_consistent == 1]

plt.figure(figsize=(7, 4))
plt.plot(valid_enrichment.threshold, valid_enrichment.matched_failure_fraction, marker='o')
plt.xlabel('maximum normalized late-residual threshold')
plt.ylabel('matched-sample logical-wrong fraction')
plt.title('Syndrome-valid case-control enrichment; not population probability')

plt.show()


## Experiments 3 and 4 — hidden instability and unstable subgraph

A hidden unstable variable must have normalized belief movement at most `hidden_belief_threshold` and normalized incoming-message residual at least `hidden_message_threshold`; this avoids selecting numerical noise by rank alone. The unstable subgraph uses the top 10% of normalized `c2v` late residuals subject to an absolute floor. Cycle counts below refer to a cycle basis, not exhaustive enumeration.

Compare posterior-belief movement with individual message movement, then inspect where unstable messages lie on the graph. This tests whether cancellation can hide active messages behind apparently stable beliefs.


In [ ]:
message_fig, message_axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
hidden = []
for message_ax, (name, message_x) in zip(message_axes, representatives.items()):
    vm = message_x['variables'].copy()
    bx = message_config['hidden_belief_threshold']
    my = message_config['hidden_message_threshold']
    vm['hidden_unstable'] = (vm.normalized_belief_instability <= bx) & (vm.normalized_message_instability >= my)
    vm['case'] = name
    hidden.append(vm)

    sc = message_ax.scatter(vm.normalized_belief_instability, vm.normalized_message_instability, c=vm.late_extrinsic_conflict, cmap='viridis', s=45)
    for message_row in vm[vm.hidden_unstable].itertuples():
        message_ax.annotate(f'v{message_row.variable_id}', (message_row.normalized_belief_instability, message_row.normalized_message_instability))
    message_ax.axvline(bx, color='grey', ls='--')
    message_ax.axhline(my, color='grey', ls='--')
    message_ax.set(title=name, xlabel='normalized belief instability', ylabel='normalized message instability')
    message_fig.colorbar(sc, ax=message_ax, label='extrinsic conflict')

hidden = pd.concat(hidden, ignore_index=True)
hidden.to_csv(message_data_paths.output('representative_hidden_variables_csv'), index=False)
plt.show()
hidden_summary = sample_metrics.groupby(['bp_syndrome_consistent', 'bp_logical_failure'])[['hidden_variable_count', 'hidden_variable_fraction']].agg(['count', 'mean', 'median', 'std']).reset_index()
hidden_summary.columns = ['_'.join((str(x) for x in col if str(x))) if isinstance(col, tuple) else col for col in hidden_summary.columns]

hidden_summary.to_csv(message_data_paths.output('hidden_instability_group_summary_csv'), index=False)
plt.figure(figsize=(9, 4))
sns.boxplot(
    data=plot_data,
    x='outcome',
    y='hidden_variable_fraction',
    showfliers=False,
)
plt.xticks(rotation=15)
plt.title('Hidden-message-instability fraction by outcome and syndrome validity')

plt.show()
message_fig, message_axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
graph_rows = []

for message_ax, (name, message_x) in zip(message_axes, representatives.items()):
    em = message_x['c2v']
    quantile_cutoff = em.normalized_late_residual.quantile(message_config['unstable_edge_quantile'])
    message_threshold = max(quantile_cutoff, message_config['unstable_normalized_residual_floor'])
    unstable = em[em.normalized_late_residual >= message_threshold]
    G = nx.Graph()

    for message_row in unstable.itertuples():
        G.add_edge(('c', int(message_row.check_id)), ('v', int(message_row.variable_id)), late_residual=float(message_row.late_residual))
    comps = list(nx.connected_components(G))
    basis = nx.cycle_basis(G)
    uv = {node[1] for node in G if node[0] == 'v'}
    error = set(np.flatnonzero(message_bit_array(message_x['row'].err)))
    message_correction = set(np.flatnonzero(message_x['trace'].hard[-1]))
    logical_vars = set(np.flatnonzero(np.any(message_x['context'].L, axis=0)))
    graph_rows.append({
        'case': name,
        'quantile_cutoff': quantile_cutoff,
        'applied_normalized_residual_threshold': message_threshold,
        'unstable_edges': G.number_of_edges(),
        'unstable_variables': len(uv),
        'unstable_checks': sum((n[0] == 'c' for n in G)),
        'component_count': len(comps),
        'largest_component_nodes': max(map(len, comps), default=0),
        'cycle_rank': G.number_of_edges() - G.number_of_nodes() + len(comps),
        'cycle_basis_4': sum((len(c) == 4 for c in basis)),
        'cycle_basis_6': sum((len(c) == 6 for c in basis)),
        'cycle_basis_8': sum((len(c) == 8 for c in basis)),
        'error_support_overlap': len(uv & error),
        'final_correction_overlap': len(uv & message_correction),
        'logical_support_overlap': len(uv & logical_vars),
    })
    pos = nx.spring_layout(G, seed=message_config['random_seed'], weight=None)
    colors = ['tab:red' if n[0] == 'c' else 'tab:blue' for n in G]
    nx.draw_networkx(
        G,
        pos=pos,
        ax=message_ax,
        node_color=colors,
        labels={n: f'{n[0]}{n[1]}' for n in G},
        node_size=600,
        font_size=8,
    )
    message_ax.set_title(f'{name}: unstable Tanner subgraph')
    message_ax.axis('off')

graph_summary = pd.DataFrame(graph_rows)
graph_summary.to_csv(message_data_paths.output('representative_unstable_subgraph_summary_csv'), index=False)
plt.show()
graph_summary


## Interpretation guardrails

- These results establish association, not causality.
- A stable logical failure falsifies the strong claim that failure necessarily implies persistent oscillation.
- Forced iterations describe post-stop dynamics and do not redefine benchmark accuracy.
- The matched analysis deduplicates `(p, syndrome, true label)` and does not report an independence-based significance test.
- Cross-$p$ plots use prior-normalized residual because raw LLR scale changes with $p$.

# Bp Stability Syndrome Logical
Configuration: `stages/01_message_dynamics/config`. Figures remain local; datasets are under `experiment/data/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
bp_stability_syndrome_logical_experiment_dir = ROOT / 'experiment/research/bp_failure_dynamics/stages/01_message_dynamics'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

bp_stability_syndrome_logical_data_paths = ExperimentPaths(ROOT, bp_stability_syndrome_logical_experiment_dir / 'config/paths.json')


# BP stability, syndrome consistency, and logical correctness

This section compares BP stability, syndrome consistency and logical correctness using trajectory diagnostics and AUROC. Benchmark decoding is the first syndrome-valid hard decision within 60 iterations; diagnostic trajectories run without early stopping. The statistical unit is the deduplicated decoder input `(p, syndrome, true_label)`. Figures are displayed inline and CSV files are written under `experiment/research/bp_failure_dynamics/stages/01_message_dynamics/`.

The primary success criterion is syndrome validity plus logical correctness. Exact equality between the decoded DEM-fault vector and one sampled physical support is recorded separately because multiple supports may define the same decoder input.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import rankdata
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5
from ldpc import BpDecoder
from noises_ca import CircuitBuilder, ErrorRate

bp_stability_syndrome_logical_config = json.loads((bp_stability_syndrome_logical_experiment_dir / 'config/bp_stability_syndrome_logical.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# Expected relationship: benchmark_iterations >= diagnostic_iterations >= 3.
# Expected relationship: robustness_iterations >= benchmark_iterations.
# ms_scaling_factor: positive.
# late_fraction: in (0, 1].
# message_zero_tolerance: non-negative.
# stability_thresholds: each item is positive.
# threshold_quantile_count: at least 2.
# validation_cases: positive.
# native_validation_tolerance: positive.
# diagnostic_candidate_limit: None or positive.
# matched_cases_per_class: positive.
bp_stability_syndrome_logical_data = bp_stability_syndrome_logical_data_paths.run()

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(bp_stability_syndrome_logical_data_paths, bp_stability_syndrome_logical_config)

bp_stability_syndrome_logical_input = bp_stability_syndrome_logical_data_paths.input('error_weight')
bp_stability_syndrome_logical_stats = bp_stability_syndrome_logical_data_paths.stats()
bp_stability_syndrome_logical_trace = bp_stability_syndrome_logical_data_paths.trace()
bp_stability_syndrome_logical_stats.mkdir(parents=True, exist_ok=True)
bp_stability_syndrome_logical_trace.mkdir(parents=True, exist_ok=True)

from experiment.utils import misc

def bp_stability_syndrome_logical_log(level, payload):
    return misc.log(level, payload, CONFIG=bp_stability_syndrome_logical_config)

sns.set_theme(style='whitegrid')
bp_stability_syndrome_logical_log(2, {
    'root': str(ROOT),
    'stats': str(bp_stability_syndrome_logical_stats),
    'trace': str(bp_stability_syndrome_logical_trace),
})


In [ ]:
bp_stability_syndrome_logical_dtype = {
    'p': float,
    'err': str,
    'det': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

bp_stability_syndrome_logical_weight2 = pd.read_csv(bp_stability_syndrome_logical_data_paths.input('weight_2_data_csv'), dtype=bp_stability_syndrome_logical_dtype)
bp_stability_syndrome_logical_weight3 = pd.read_csv(bp_stability_syndrome_logical_data_paths.input('weight_3_data_csv'), dtype=bp_stability_syndrome_logical_dtype)

required = set(bp_stability_syndrome_logical_dtype)
for bp_stability_syndrome_logical_frame, weight in ((bp_stability_syndrome_logical_weight2, 2), (bp_stability_syndrome_logical_weight3, 3)):
    bp_stability_syndrome_logical_frame['syndrome_weight'] = bp_stability_syndrome_logical_frame.det.str.count('1').astype(int)
input_audit = pd.DataFrame([{
    'weight2_rows': len(bp_stability_syndrome_logical_weight2),
    'weight3_rows': len(bp_stability_syndrome_logical_weight3),
    'weight2_unique_decoder_inputs': len(bp_stability_syndrome_logical_weight2.drop_duplicates(['p', 'det', 'true'])),
    'weight3_unique_decoder_inputs': len(bp_stability_syndrome_logical_weight3.drop_duplicates(['p', 'det', 'true'])),
}])

input_audit.to_csv(bp_stability_syndrome_logical_data_paths.output('stability_input_audit_csv'), index=False)
input_audit


## Instrumented parallel minimum-sum BP

Index 0 is initialization. Cross-$p$ comparisons use residual divided by the absolute prior LLR. A 60-iteration forced trace contains the identical first 50 diagnostic updates and the complete 60-iteration benchmark search; benchmark classification still selects the first syndrome-valid state rather than the forced final state.

In [ ]:
from experiment.utils import bp_message_dynamics

bp_stability_syndrome_logical_Context = bp_message_dynamics.Context

@dataclass(frozen=True)
class bp_stability_syndrome_logical_MessageTrace:
    c2v: np.ndarray
    v2c: np.ndarray
    llr: np.ndarray
    hard: np.ndarray
    syndrome_consistent: np.ndarray
    first_valid_iteration: int | None

bp_stability_syndrome_logical_cache = {}

def bp_stability_syndrome_logical_context_for(p):
    return bp_message_dynamics.context_for(
        p,
        CACHE=bp_stability_syndrome_logical_cache,
        CA_9_2_5=CA_9_2_5,
        CircuitBuilder=CircuitBuilder,
        Context=bp_stability_syndrome_logical_Context,
        ErrorRate=ErrorRate,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
    )

from experiment.utils import misc

bp_stability_syndrome_logical_bit_array = misc.bit_array
bp_stability_syndrome_logical_bit_text = misc.bit_text
bp_stability_syndrome_logical_logical = bp_message_dynamics.logical

def bp_stability_syndrome_logical_trace_min_sum(
    context,
    syndrome,
    iterations,
    stop_early=False,
):
    return bp_message_dynamics.trace_min_sum(context, syndrome, iterations, stop_early, CONFIG=bp_stability_syndrome_logical_config, MessageTrace=bp_stability_syndrome_logical_MessageTrace)

def trace_prefix(trace, iterations):
    assert 0 < iterations < len(trace.llr)
    end = iterations + 1
    valid_indices = np.flatnonzero(trace.syndrome_consistent[:end])
    first = None if valid_indices.size == 0 else int(valid_indices[0])

    return bp_stability_syndrome_logical_MessageTrace(trace.c2v[:end], trace.v2c[:end], trace.llr[:end], trace.hard[:end], trace.syndrome_consistent[:end], first)

def benchmark_state(trace):
    index = trace.first_valid_iteration if trace.first_valid_iteration is not None else bp_stability_syndrome_logical_config['benchmark_iterations']
    assert index <= bp_stability_syndrome_logical_config['benchmark_iterations'] and index < len(trace.hard)
    return (index, trace.hard[index], bool(trace.syndrome_consistent[index]))

def bp_stability_syndrome_logical_edge_metrics(context, trace):
    return bp_message_dynamics.edge_metrics(context, trace, CONFIG=bp_stability_syndrome_logical_config)

def bp_stability_syndrome_logical_variable_metrics(context, trace, edges):
    return bp_message_dynamics.variable_metrics(context, trace, edges, CONFIG=bp_stability_syndrome_logical_config)

def classify_outcome(syndrome_consistent, logical_failure):
    labels = {
        (False, True): 'inconsistent-wrong',
        (False, False): 'inconsistent-correct',
        (True, False): 'consistent-correct',
        (True, True): 'consistent-wrong',
    }

    return labels[bool(syndrome_consistent), bool(logical_failure)]

METRICS = ['max_normalized_late_residual', 'mean_normalized_late_residual', 'max_flip_rate', 'mean_flip_rate', 'max_period2_strength', 'mean_message_instability', 'mean_belief_instability', 'mean_extrinsic_conflict']


## Native validation

Before interpreting any trace, compare the instrumented trajectory with the native decoding reference. This checks that the diagnostics describe the intended decoder rather than an unrelated Python implementation.


In [ ]:
validation_pool = bp_stability_syndrome_logical_weight2.drop_duplicates(['p', 'det'])
assert len(validation_pool) >= bp_stability_syndrome_logical_config['validation_cases']
bp_stability_syndrome_logical_validation = validation_pool.sample(bp_stability_syndrome_logical_config['validation_cases'], random_state=bp_stability_syndrome_logical_config['random_seed'])
validation_rows = []

for bp_stability_syndrome_logical_row in tqdm(bp_stability_syndrome_logical_validation.itertuples(index=False), total=len(bp_stability_syndrome_logical_validation), desc='Validate native BP', disable=bp_stability_syndrome_logical_config['print_level'] == 1):
    context = bp_stability_syndrome_logical_context_for(bp_stability_syndrome_logical_row.p)
    bp_stability_syndrome_logical_syndrome = bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.det)
    trace = bp_stability_syndrome_logical_trace_min_sum(context, bp_stability_syndrome_logical_syndrome, bp_stability_syndrome_logical_config['benchmark_iterations'])
    bp_stability_syndrome_logical_native = BpDecoder(
        context.H,
        error_channel=context.priors,
        max_iter=bp_stability_syndrome_logical_config['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=bp_stability_syndrome_logical_config['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    native_hard = np.asarray(bp_stability_syndrome_logical_native.decode(bp_stability_syndrome_logical_syndrome), np.uint8)
    native_iteration = int(bp_stability_syndrome_logical_native.iter)
    assert 0 <= native_iteration <= bp_stability_syndrome_logical_config['benchmark_iterations']
    native_llr = np.asarray(bp_stability_syndrome_logical_native.log_prob_ratios)
    np.testing.assert_allclose(trace.llr[native_iteration], native_llr, rtol=bp_stability_syndrome_logical_config['native_validation_tolerance'], atol=bp_stability_syndrome_logical_config['native_validation_tolerance'])
    hard_difference = trace.hard[native_iteration] != native_hard

    if hard_difference.any():
        assert np.max(np.abs(trace.llr[native_iteration, hard_difference])) <= bp_stability_syndrome_logical_config['native_validation_tolerance']
        assert np.max(np.abs(native_llr[hard_difference])) <= bp_stability_syndrome_logical_config['native_validation_tolerance']
    np.testing.assert_array_equal(context.H @ native_hard % 2, bp_stability_syndrome_logical_syndrome) if bp_stability_syndrome_logical_native.converge else None
    validation_rows.append({
        'p': bp_stability_syndrome_logical_row.p,
        'syndrome': bp_stability_syndrome_logical_row.det,
        'native_iterations': native_iteration,
        'instrumented_first_valid_iteration': trace.first_valid_iteration,
        'syndrome_consistent': bp_stability_syndrome_logical_native.converge,
        'near_zero_hard_tie_count': int(hard_difference.sum()),
        'max_abs_posterior_difference': float(np.max(np.abs(trace.llr[native_iteration] - native_llr))),
    })

validation_frame = pd.DataFrame(validation_rows)
paired_iterations = validation_frame.dropna(subset=['instrumented_first_valid_iteration'])
validation_summary = pd.DataFrame([{
    'cases': len(validation_frame),
    'native_syndrome_consistent_count': int(validation_frame.syndrome_consistent.sum()),
    'near_zero_hard_tie_count': int(validation_frame.near_zero_hard_tie_count.sum()),
    'max_abs_posterior_difference': validation_frame.max_abs_posterior_difference.max(),
    'max_abs_native_vs_first_valid_iteration_difference': np.max(np.abs(paired_iterations.native_iterations - paired_iterations.instrumented_first_valid_iteration)),
}])

validation_summary.to_csv(bp_stability_syndrome_logical_data_paths.output('instrumented_native_validation_summary_csv'), index=False)
validation_summary


## Diagnostic population

The unbalanced population is deduplicated by `(p, syndrome, true_label)`. Physical-support multiplicity is retained. Benchmark labels use the first valid state; message metrics use only the first 50 forced updates.

In [ ]:
identity = ['p', 'det', 'true']
support_multiplicity = bp_stability_syndrome_logical_weight3.groupby(identity).err.nunique().rename('physical_support_multiplicity')
population = bp_stability_syndrome_logical_weight3.drop_duplicates(identity).sample(frac=1, random_state=bp_stability_syndrome_logical_config['random_seed'])

if bp_stability_syndrome_logical_config['diagnostic_candidate_limit'] is not None:
    population = population.head(bp_stability_syndrome_logical_config['diagnostic_candidate_limit'])
population = population.join(support_multiplicity, on=identity).reset_index(drop=True)
bp_stability_syndrome_logical_records = []

for bp_stability_syndrome_logical_sample_id, bp_stability_syndrome_logical_row in enumerate(tqdm(population.itertuples(index=False), total=len(population), desc='Diagnostic population', disable=bp_stability_syndrome_logical_config['print_level'] == 1)):
    context = bp_stability_syndrome_logical_context_for(bp_stability_syndrome_logical_row.p)
    bp_stability_syndrome_logical_syndrome = bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.det)
    true_label = bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.true)
    sampled_error = bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.err)
    assert sampled_error.size == context.H.shape[1] and np.array_equal(context.H @ sampled_error % 2, bp_stability_syndrome_logical_syndrome)
    full_trace = bp_stability_syndrome_logical_trace_min_sum(context, bp_stability_syndrome_logical_syndrome, bp_stability_syndrome_logical_config['benchmark_iterations'])
    diagnostic = trace_prefix(full_trace, bp_stability_syndrome_logical_config['diagnostic_iterations'])
    benchmark_iteration, bp_stability_syndrome_logical_correction, syndrome_consistent = benchmark_state(full_trace)
    prediction = bp_stability_syndrome_logical_logical(context, bp_stability_syndrome_logical_correction)
    logical_failure = not np.array_equal(prediction, true_label)
    edges = bp_stability_syndrome_logical_edge_metrics(context, diagnostic)
    variables = bp_stability_syndrome_logical_variable_metrics(context, diagnostic, edges)
    residual_error = bp_stability_syndrome_logical_correction ^ sampled_error
    bp_stability_syndrome_logical_records.append({
        'sample_id': bp_stability_syndrome_logical_sample_id,
        'p': bp_stability_syndrome_logical_row.p,
        'error_weight': 3,
        'error_support': bp_stability_syndrome_logical_row.err,
        'physical_support_multiplicity': int(bp_stability_syndrome_logical_row.physical_support_multiplicity),
        'syndrome': bp_stability_syndrome_logical_row.det,
        'syndrome_weight': bp_stability_syndrome_logical_row.syndrome_weight,
        'true_label': bp_stability_syndrome_logical_row.true,
        'benchmark_iteration': benchmark_iteration,
        'benchmark_prediction': bp_stability_syndrome_logical_bit_text(prediction),
        'bp_syndrome_consistent': int(syndrome_consistent),
        'bp_logical_failure': int(logical_failure),
        'decoder_success': int(syndrome_consistent and (not logical_failure)),
        'outcome_class': classify_outcome(syndrome_consistent, logical_failure),
        'correction_equals_sampled_error': int(np.array_equal(bp_stability_syndrome_logical_correction, sampled_error)),
        'residual_error_weight': int(residual_error.sum()),
        'max_normalized_late_residual': edges.normalized_late_residual.max(),
        'mean_normalized_late_residual': edges.normalized_late_residual.mean(),
        'max_flip_rate': edges.sign_flip_rate.max(),
        'mean_flip_rate': edges.sign_flip_rate.mean(),
        'max_period2_strength': edges.period2_strength.max(),
        'mean_message_instability': variables.message_instability.mean(),
        'mean_belief_instability': variables.belief_instability.mean(),
        'mean_extrinsic_conflict': variables.extrinsic_conflict.mean(),
    })

diagnostic_population = pd.DataFrame(bp_stability_syndrome_logical_records)
diagnostic_population.to_csv(bp_stability_syndrome_logical_data_paths.output('diagnostic_population_metrics_csv'), index=False)
diagnostic_population[['sample_id', 'p', 'syndrome', 'true_label', 'error_support', 'physical_support_multiplicity', 'outcome_class', 'correction_equals_sampled_error', 'residual_error_weight']].to_csv(bp_stability_syndrome_logical_data_paths.output('exact_error_match_audit_csv'), index=False)
diagnostic_population.groupby('outcome_class').size().rename('cases').to_frame()


## Experiments A and B — syndrome validity and four outcome classes

The next calculations cross syndrome consistency with logical correctness. Keeping all four groups visible prevents a satisfied parity-check constraint from being treated as proof of a correct logical sector.


In [ ]:
diagnostic_population = pd.read_csv(bp_stability_syndrome_logical_data_paths.output('diagnostic_population_metrics_csv'), dtype={
    'p': float,
    'syndrome': str,
    'true_label': str,
    'error_support': str,
    'outcome_class': str,
})


In [ ]:
primary = 'max_normalized_late_residual'
bp_stability_syndrome_logical_thresholds = np.unique(np.quantile(diagnostic_population[primary], np.linspace(0, 0.95, bp_stability_syndrome_logical_config['threshold_quantile_count'])))
threshold_rows = []

for bp_stability_syndrome_logical_threshold in bp_stability_syndrome_logical_thresholds:
    bp_stability_syndrome_logical_selected = diagnostic_population[diagnostic_population[primary] > bp_stability_syndrome_logical_threshold]
    if len(bp_stability_syndrome_logical_selected):
        threshold_rows.append({
            'threshold': bp_stability_syndrome_logical_threshold,
            'cases': len(bp_stability_syndrome_logical_selected),
            'syndrome_consistent_count': int(bp_stability_syndrome_logical_selected.bp_syndrome_consistent.sum()),
            'syndrome_consistent_fraction': bp_stability_syndrome_logical_selected.bp_syndrome_consistent.mean(),
        })

syndrome_curve = pd.DataFrame(threshold_rows)
syndrome_curve.to_csv(bp_stability_syndrome_logical_data_paths.output('syndrome_consistent_vs_residual_threshold_csv'), index=False)
summary_rows = []

for outcome, bp_stability_syndrome_logical_group in diagnostic_population.groupby('outcome_class', observed=True):
    for bp_stability_syndrome_logical_metric in METRICS:
        bp_stability_syndrome_logical_values = bp_stability_syndrome_logical_group[bp_stability_syndrome_logical_metric]
        summary_rows.append({
            'outcome_class': outcome,
            'metric': bp_stability_syndrome_logical_metric,
            'count': len(bp_stability_syndrome_logical_values),
            'mean': bp_stability_syndrome_logical_values.mean(),
            'median': bp_stability_syndrome_logical_values.median(),
            'std': bp_stability_syndrome_logical_values.std(),
            'q25': bp_stability_syndrome_logical_values.quantile(0.25),
            'q75': bp_stability_syndrome_logical_values.quantile(0.75),
        })

four_summary = pd.DataFrame(summary_rows)
four_summary.to_csv(bp_stability_syndrome_logical_data_paths.output('four_outcome_group_summary_csv'), index=False)
separation_rows = []
valid = diagnostic_population[diagnostic_population.bp_syndrome_consistent == 1]

for bp_stability_syndrome_logical_metric in METRICS:
    invalid_values = diagnostic_population.loc[diagnostic_population.bp_syndrome_consistent == 0, bp_stability_syndrome_logical_metric]
    valid_values = valid[bp_stability_syndrome_logical_metric]
    correct_values = valid.loc[valid.bp_logical_failure == 0, bp_stability_syndrome_logical_metric]
    wrong_values = valid.loc[valid.bp_logical_failure == 1, bp_stability_syndrome_logical_metric]
    separation_rows.append({
        'metric': bp_stability_syndrome_logical_metric,
        'invalid_mean': invalid_values.mean(),
        'valid_mean': valid_values.mean(),
        'delta_syn_mean': invalid_values.mean() - valid_values.mean(),
        'valid_correct_mean': correct_values.mean(),
        'valid_wrong_mean': wrong_values.mean(),
        'delta_log_mean': wrong_values.mean() - correct_values.mean(),
        'invalid_median': invalid_values.median(),
        'valid_median': valid_values.median(),
        'delta_syn_median': invalid_values.median() - valid_values.median(),
        'valid_correct_median': correct_values.median(),
        'valid_wrong_median': wrong_values.median(),
        'delta_log_median': wrong_values.median() - correct_values.median(),
    })

separation = pd.DataFrame(separation_rows)
separation.to_csv(bp_stability_syndrome_logical_data_paths.output('syndrome_vs_logical_separation_csv'), index=False)
separation


In [ ]:
bp_stability_syndrome_logical_fig, bp_stability_syndrome_logical_axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
order = ['inconsistent-wrong', 'inconsistent-correct', 'consistent-correct', 'consistent-wrong']
for bp_stability_syndrome_logical_ax, bp_stability_syndrome_logical_metric in zip(bp_stability_syndrome_logical_axes, ['max_normalized_late_residual', 'max_flip_rate', 'max_period2_strength']):
    sns.boxplot(
        data=diagnostic_population,
        x='outcome_class',
        y=bp_stability_syndrome_logical_metric,
        order=order,
        showfliers=False,
        ax=bp_stability_syndrome_logical_ax,
    )
    sns.stripplot(
        data=diagnostic_population.sample(min(2000, len(diagnostic_population)), random_state=bp_stability_syndrome_logical_config['random_seed']),
        x='outcome_class',
        y=bp_stability_syndrome_logical_metric,
        order=order,
        alpha=0.15,
        size=2,
        color='black',
        ax=bp_stability_syndrome_logical_ax,
    )
    bp_stability_syndrome_logical_ax.tick_params(axis='x', rotation=25)

plt.show()


## Experiments C and D — conditional logical enrichment and stable logical failures

Restrict the comparison to syndrome-consistent cases and measure whether residuals still separate correct from wrong logical sectors. The threshold scan also counts logically wrong endpoints whose messages are already stable.


In [ ]:
bp_stability_syndrome_logical_paired_rows = []
bp_stability_syndrome_logical_pair_id = 0
for bp_stability_syndrome_logical_, bp_stability_syndrome_logical_group in valid.groupby(['p', 'syndrome_weight']):
    wrong = bp_stability_syndrome_logical_group[bp_stability_syndrome_logical_group.bp_logical_failure == 1].sample(frac=1, random_state=bp_stability_syndrome_logical_config['random_seed']).reset_index(drop=True)
    correct = bp_stability_syndrome_logical_group[bp_stability_syndrome_logical_group.bp_logical_failure == 0].sample(frac=1, random_state=bp_stability_syndrome_logical_config['random_seed'] + 1).reset_index(drop=True)
    for index in range(min(len(wrong), len(correct))):
        bp_stability_syndrome_logical_paired_rows.extend([{
            **wrong.iloc[index].to_dict(),
            'matched_pair_id': bp_stability_syndrome_logical_pair_id,
        }, {
            **correct.iloc[index].to_dict(),
            'matched_pair_id': bp_stability_syndrome_logical_pair_id,
        }])
        bp_stability_syndrome_logical_pair_id += 1

bp_stability_syndrome_logical_matched = pd.DataFrame(bp_stability_syndrome_logical_paired_rows)
pair_ids = pd.Series(bp_stability_syndrome_logical_matched.matched_pair_id.unique())
if len(pair_ids) > bp_stability_syndrome_logical_config['matched_cases_per_class']:
    keep_ids = pair_ids.sample(bp_stability_syndrome_logical_config['matched_cases_per_class'], random_state=bp_stability_syndrome_logical_config['random_seed'])
    bp_stability_syndrome_logical_matched = bp_stability_syndrome_logical_matched[bp_stability_syndrome_logical_matched.matched_pair_id.isin(keep_ids)]
matched_thresholds = np.unique(np.quantile(bp_stability_syndrome_logical_matched[primary], np.linspace(0, 0.95, bp_stability_syndrome_logical_config['threshold_quantile_count'])))
enrichment_rows = []

for bp_stability_syndrome_logical_threshold in matched_thresholds:
    bp_stability_syndrome_logical_selected = bp_stability_syndrome_logical_matched[bp_stability_syndrome_logical_matched[primary] > bp_stability_syndrome_logical_threshold]
    if len(bp_stability_syndrome_logical_selected):
        enrichment_rows.append({
            'threshold': bp_stability_syndrome_logical_threshold,
            'cases': len(bp_stability_syndrome_logical_selected),
            'matched_sample_logical_wrong_fraction': bp_stability_syndrome_logical_selected.bp_logical_failure.mean(),
            'matched_baseline': 0.5,
        })

matched_enrichment = pd.DataFrame(enrichment_rows)
matched_enrichment.to_csv(bp_stability_syndrome_logical_data_paths.output('matched_logical_wrong_enrichment_csv'), index=False)
matched_enrichment


In [ ]:
stable_rows = []
for bp_stability_syndrome_logical_threshold in bp_stability_syndrome_logical_config['stability_thresholds']:
    correct = valid[valid.bp_logical_failure == 0]
    wrong = valid[valid.bp_logical_failure == 1]
    stable_rows.append({
        'threshold': bp_stability_syndrome_logical_threshold,
        'consistent_correct_count': len(correct),
        'consistent_correct_stable_fraction': (correct[primary] < bp_stability_syndrome_logical_threshold).mean(),
        'consistent_wrong_count': len(wrong),
        'consistent_wrong_stable_fraction': (wrong[primary] < bp_stability_syndrome_logical_threshold).mean(),
    })

stable_fraction = pd.DataFrame(stable_rows)
stable_fraction.to_csv(bp_stability_syndrome_logical_data_paths.output('stable_valid_logical_wrong_fraction_csv'), index=False)
stable_fraction


In [ ]:
bp_stability_syndrome_logical_fig, bp_stability_syndrome_logical_axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
bp_stability_syndrome_logical_axes[0].plot(matched_enrichment.threshold, matched_enrichment.matched_sample_logical_wrong_fraction, marker='o')
bp_stability_syndrome_logical_axes[0].axhline(0.5, color='black', linestyle='--')
bp_stability_syndrome_logical_axes[0].set(xlabel='maximum normalized late-residual threshold', ylabel='logical-wrong fraction')

bp_stability_syndrome_logical_axes[1].plot(stable_fraction.threshold, stable_fraction.consistent_correct_stable_fraction, marker='o', label='consistent-correct')
bp_stability_syndrome_logical_axes[1].plot(stable_fraction.threshold, stable_fraction.consistent_wrong_stable_fraction, marker='o', label='consistent-wrong')
bp_stability_syndrome_logical_axes[1].set_xscale('log')
bp_stability_syndrome_logical_axes[1].set(xlabel='stability threshold', ylabel='fraction classified as stable')
bp_stability_syndrome_logical_axes[1].legend()

plt.show()


## Experiments E and F — all unique weight-2 benchmark failures

Inspect every unique weight-2 benchmark failure rather than selecting only an illustrative oscillating case. Compare the benchmark stopping outcome with the longer diagnostic trajectory.


In [ ]:
weight2_unique = bp_stability_syndrome_logical_weight2.drop_duplicates(['p', 'det', 'true']).reset_index(drop=True)
failure_inputs = []
for bp_stability_syndrome_logical_row in tqdm(weight2_unique.itertuples(index=False), total=len(weight2_unique), desc='Audit weight-2 benchmark', disable=bp_stability_syndrome_logical_config['print_level'] == 1):
    context = bp_stability_syndrome_logical_context_for(bp_stability_syndrome_logical_row.p)
    trace = bp_stability_syndrome_logical_trace_min_sum(context, bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.det), bp_stability_syndrome_logical_config['benchmark_iterations'], stop_early=True)
    prediction = bp_stability_syndrome_logical_logical(context, trace.hard[-1])
    valid_state = bool(trace.syndrome_consistent[-1])

    if valid_state and (not np.array_equal(prediction, bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.true))):
        failure_inputs.append(bp_stability_syndrome_logical_row)

assert len(failure_inputs) > 0

def trajectory_arrays(context, trace, true_label):
    predictions = [bp_stability_syndrome_logical_bit_text(bp_stability_syndrome_logical_logical(context, hard)) for hard in trace.hard]
    correct = np.asarray([prediction == bp_stability_syndrome_logical_bit_text(true_label) for prediction in predictions], bool)
    return (predictions, correct)

def classify_logical_trajectory(valid_history, correct_history):
    valid_indices = np.flatnonzero(valid_history)
    assert valid_indices.size > 0
    first = int(valid_indices[0])
    later_valid = valid_indices[valid_indices > first]

    if later_valid.size == 0:
        return 'transient-valid_only'
    later_correct = later_valid[correct_history[later_valid]]

    if later_correct.size == 0:
        return 'persistent_wrong-valid'
    first_correct = int(later_correct[0])

    if np.any(valid_history[first_correct + 1:] & ~correct_history[first_correct + 1:]):
        return 'logical-sector-switching'

    return 'wrong-valid_to_correct-valid'

trajectory_rows = []
iteration_rows = []
plot_records = []

for bp_stability_syndrome_logical_sample_id, bp_stability_syndrome_logical_row in enumerate(tqdm(failure_inputs, desc='Weight-2 trajectories', disable=bp_stability_syndrome_logical_config['print_level'] == 1)):
    context = bp_stability_syndrome_logical_context_for(bp_stability_syndrome_logical_row.p)
    bp_stability_syndrome_logical_syndrome = bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.det)
    true_label = bp_stability_syndrome_logical_bit_array(bp_stability_syndrome_logical_row.true)
    trace100 = bp_stability_syndrome_logical_trace_min_sum(context, bp_stability_syndrome_logical_syndrome, bp_stability_syndrome_logical_config['robustness_iterations'])
    trace50 = trace_prefix(trace100, bp_stability_syndrome_logical_config['diagnostic_iterations'])
    prediction100, correct100 = trajectory_arrays(context, trace100, true_label)
    prediction50, correct50 = trajectory_arrays(context, trace50, true_label)
    first = trace100.first_valid_iteration
    assert first is not None and first <= bp_stability_syndrome_logical_config['benchmark_iterations'] and (not correct100[first])
    later50 = np.flatnonzero(trace50.syndrome_consistent & correct50)
    later50 = later50[later50 > first]
    later100 = np.flatnonzero(trace100.syndrome_consistent & correct100)
    later100 = later100[later100 > first]
    trajectory_rows.append({
        'sample_id': bp_stability_syndrome_logical_sample_id,
        'p': bp_stability_syndrome_logical_row.p,
        'syndrome': bp_stability_syndrome_logical_row.det,
        'true_label': bp_stability_syndrome_logical_row.true,
        'first_valid_iteration': first,
        'first_valid_prediction': prediction100[first],
        'first_valid_correct': False,
        'later_correct_valid_exists': bool(later50.size),
        'first_later_correct_valid_iteration': np.nan if later50.size == 0 else int(later50[0]),
        'final_syndrome_consistent': bool(trace50.syndrome_consistent[-1]),
        'final_prediction': prediction50[-1],
        'final_logical_correct': bool(correct50[-1]),
        'trajectory_type': classify_logical_trajectory(trace50.syndrome_consistent, correct50),
        'robustness_later_correct_valid_exists': bool(later100.size),
        'robustness_first_later_correct_valid_iteration': np.nan if later100.size == 0 else int(later100[0]),
        'robustness_final_syndrome_consistent': bool(trace100.syndrome_consistent[-1]),
        'robustness_final_prediction': prediction100[-1],
        'robustness_final_logical_correct': bool(correct100[-1]),
        'robustness_trajectory_type': classify_logical_trajectory(trace100.syndrome_consistent, correct100),
    })
    changes = np.r_[False, np.any(trace100.hard[1:] != trace100.hard[:-1], axis=1)]
    residuals = np.abs(np.diff(trace100.c2v, axis=0))

    for iteration in range(len(trace100.hard)):
        iteration_rows.append({
            'sample_id': bp_stability_syndrome_logical_sample_id,
            'iteration': iteration,
            'syndrome_consistent': bool(trace100.syndrome_consistent[iteration]),
            'logical_prediction': prediction100[iteration],
            'logical_correct': bool(correct100[iteration]),
            'hard_weight': int(trace100.hard[iteration].sum()),
            'hard_decision_changed': bool(changes[iteration]),
            'max_edge_residual_at_iteration': np.nan if iteration == 0 else residuals[iteration - 1].max(),
            'mean_edge_residual_at_iteration': np.nan if iteration == 0 else residuals[iteration - 1].mean(),
            'within_primary_50_iterations': iteration <= bp_stability_syndrome_logical_config['diagnostic_iterations'],
        })

    plot_records.append((bp_stability_syndrome_logical_sample_id, bp_stability_syndrome_logical_row, trace100, prediction100, correct100, later100))

bp_stability_syndrome_logical_trajectory_summary = pd.DataFrame(trajectory_rows)
iteration_trajectories = pd.DataFrame(iteration_rows)
assert len(bp_stability_syndrome_logical_trajectory_summary) == len(failure_inputs) and iteration_trajectories.groupby('sample_id').size().eq(bp_stability_syndrome_logical_config['robustness_iterations'] + 1).all()

bp_stability_syndrome_logical_trajectory_summary.to_csv(bp_stability_syndrome_logical_data_paths.output('weight2_failure_trajectory_summary_csv'), index=False)
iteration_trajectories.to_csv(bp_stability_syndrome_logical_data_paths.output('weight2_failure_iteration_trajectories_csv'), index=False)
bp_stability_syndrome_logical_trajectory_summary


In [ ]:
bp_stability_syndrome_logical_fig, bp_stability_syndrome_logical_axes = plt.subplots(len(plot_records), 1, figsize=(14, max(3, 2.7 * len(plot_records))), squeeze=False, constrained_layout=True)
for bp_stability_syndrome_logical_ax, (bp_stability_syndrome_logical_sample_id, bp_stability_syndrome_logical_row, trace, predictions, correct, later) in zip(bp_stability_syndrome_logical_axes[:, 0], plot_records):
    sectors = np.asarray([int(value, 2) for value in predictions])
    iterations = np.arange(len(sectors))
    for is_valid, marker in ((False, 'x'), (True, 's')):
        mask = trace.syndrome_consistent == is_valid
        bp_stability_syndrome_logical_ax.scatter(
            iterations[mask],
            sectors[mask],
            marker=marker,
            c=np.where(correct[mask], 'tab:green', 'tab:red'),
            s=20,
            alpha=0.75,
        )

    bp_stability_syndrome_logical_ax.axvline(trace.first_valid_iteration, color='black', linestyle='--', label='first consistent')
    bp_stability_syndrome_logical_ax.annotate(
        trace.first_valid_iteration,
        xy=(trace.first_valid_iteration, 0),
        xycoords=('data', 'axes fraction'),
        xytext=(0, -18),
        textcoords='offset points',
        ha='center',
    )
    if later.size:
        bp_stability_syndrome_logical_ax.axvline(later[0], color='tab:green', linestyle=':', label='first consistent-correct')
        bp_stability_syndrome_logical_ax.annotate(
            later[0],
            xy=(later[0], 0),
            xycoords=('data', 'axes fraction'),
            xytext=(0, -18),
            textcoords='offset points',
            ha='center',
            color='tab:green',
        )
    bp_stability_syndrome_logical_ax.set(
        yticks=range(4),
        yticklabels=['00', '01', '10', '11'],
        ylabel=f'sample {bp_stability_syndrome_logical_sample_id}',
        xlim=(0, bp_stability_syndrome_logical_config['robustness_iterations']),
    )
    bp_stability_syndrome_logical_ax.legend(loc='upper right')

bp_stability_syndrome_logical_axes[-1, 0].set_xlabel('iteration')
plt.show()


## AUROC and main two-panel hierarchy figure

The two prediction tasks use different labels and denominators: syndrome inconsistency in the diagnostic population, and logical error conditional on consistency. Compare their AUROC values without pooling the tasks.


In [ ]:
def auc_from_scores(target, score):
    target = np.asarray(target, int)
    score = np.asarray(score, float)
    assert set(np.unique(target)) == {0, 1} and np.isfinite(score).all()
    positive = target == 1
    n_positive = int(positive.sum())
    n_negative = int((~positive).sum())
    ranks = rankdata(score, method='average')
    auc = (ranks[positive].sum() - n_positive * (n_positive + 1) / 2) / (n_positive * n_negative)

    return (float(auc), n_positive, n_negative)

auc_syn, syn_pos, syn_neg = auc_from_scores(1 - diagnostic_population.bp_syndrome_consistent, diagnostic_population[primary])
auc_log, log_pos, log_neg = auc_from_scores(valid.bp_logical_failure, valid[primary])
auc_summary = pd.DataFrame([{
    'task': 'predict-syndrome-inconsistent',
    'metric': primary,
    'auroc': auc_syn,
    'positive_cases': syn_pos,
    'negative_cases': syn_neg,
}, {
    'task': 'predict-logical-wrong-given-consistent',
    'metric': primary,
    'auroc': auc_log,
    'positive_cases': log_pos,
    'negative_cases': log_neg,
}])

auc_summary.to_csv(bp_stability_syndrome_logical_data_paths.output('residual_prediction_auc_csv'), index=False)
auc_summary


In [ ]:
bp_stability_syndrome_logical_fig, bp_stability_syndrome_logical_axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
bp_stability_syndrome_logical_axes[0].plot(syndrome_curve.threshold, syndrome_curve.syndrome_consistent_fraction, marker='o')
bp_stability_syndrome_logical_axes[0].set(xlabel='maximum normalized late-residual threshold', ylabel='syndrome-consistent fraction', title='Dynamics to syndrome consistency')

bp_stability_syndrome_logical_axes[1].plot(matched_enrichment.threshold, matched_enrichment.matched_sample_logical_wrong_fraction, marker='o')
bp_stability_syndrome_logical_axes[1].axhline(0.5, color='black', linestyle='--')
bp_stability_syndrome_logical_axes[1].set(xlabel='maximum normalized late-residual threshold', ylabel='logical-wrong fraction', title='Syndrome-consistent dynamics to logical correctness')

plt.show()


## Exact sampled-error match audit

This audit compares correction bits with the sampled physical error. Exact equality is stricter than matching logical observables, so the audit is kept separate from decoding success.


In [ ]:
exact_summary = diagnostic_population.groupby(['physical_support_multiplicity', 'outcome_class'], as_index=False).agg(
    cases=('sample_id', 'size'),
    exact_sampled_error_matches=('correction_equals_sampled_error', 'sum'),
    exact_sampled_error_match_fraction=('correction_equals_sampled_error', 'mean'),
    mean_residual_error_weight=('residual_error_weight', 'mean'),
    median_residual_error_weight=('residual_error_weight', 'median'),
)
exact_summary.to_csv(bp_stability_syndrome_logical_data_paths.output('exact_error_match_summary_csv'), index=False)
exact_summary


## Experiment G — interface limitation

Experiment G is not evaluated here. The current `AdaptiveBpDecoder` performs one main BP run followed by independently initialized complementary augmented-PCM views. It does not expose a stateful `T1` original-PCM run followed by a `T2` continuation from the same messages. Calling a plain decoder with `T1+T2` and comparing it with the current multi-view decoder would therefore mix continuation, restart, and view-count effects instead of isolating the effect of additional iterations.

## Interpretation guardrails

- These experiments measure association, not causality.
- Syndrome-invalid logical-label matches are not decoding successes.
- Forced 50/100-iteration states do not redefine benchmark performance.
- The matched enrichment curve is a case-control fraction with baseline 0.5, not a population failure probability.
- `correction_equals_sampled_error` compares with the representative support retained after decoder-input deduplication. When `physical_support_multiplicity > 1`, it is not an identifiable exact-recovery probability.